# ML-KEM-768 Benchmark Notebook

Interactive equivalent of `ml_kem_bench.py`. For each selected operation, runs N iterations with random inputs and reports:

- **HW cycles** (median / min / max) — pure accelerator latency from on-chip counter.
- **HW latency** in µs — derived from cycles at 100 MHz.
- **Wall time** seen by Python — includes register writes, cache ops, polling.
- **PYNQ overhead** — (wall − hw), the software control-path cost.
- **Throughput** in ops/s under pulse-then-poll, single thread.

## Op selection

Edit the `op` variable in the **config cell** to choose what to bench:

| `op` value | Behavior |
|---|---|
| `"all"` | Run KeyGen, Encaps, Decaps, Full KEM (default, mirrors `ml_kem_bench.py` no-flag) |
| `"keygen"` | KeyGen only, random `d, z` per iter |
| `"encaps"` | Encaps only, warm `pk` + random `m` |
| `"decaps"` | Decaps only, warm `sk` + warm `ct` (match branch) |
| `"full"` | Full KEM (KG + Enc + Dec) per iter, asserts ss round-trip match |

Equivalent CLI: `sudo python3 ml_kem_bench.py --op <op> -n <n>`.

In [ ]:
import os
import secrets
import statistics
import time

from ml_kem_driver import MLKem768, cycles_to_us


def bench_op(label, fn, n):
    """Run fn() n times. fn must return (..., cycles) or plain cycles."""
    cycles_list = []
    wall_list = []
    for _ in range(n):
        t0 = time.monotonic()
        result = fn()
        wall_list.append(time.monotonic() - t0)
        cyc = result[-1] if isinstance(result, tuple) else result
        cycles_list.append(cyc)

    c_med = statistics.median(cycles_list)
    c_min = min(cycles_list)
    c_max = max(cycles_list)
    w_med = statistics.median(wall_list)
    w_min = min(wall_list)
    w_max = max(wall_list)
    throughput = n / sum(wall_list) if sum(wall_list) > 0 else 0.0

    print(f"  {label}")
    print(f"    HW cycles   : median={int(c_med):6d}  min={c_min:6d}  max={c_max:6d}")
    print(f"    HW latency  : median={cycles_to_us(c_med):6.1f} us  (= {int(c_med)} cyc @ 100 MHz)")
    print(f"    Wall time   : median={w_med*1e6:6.1f} us  min={w_min*1e6:6.1f} us  max={w_max*1e6:6.1f} us")
    print(f"    PYNQ ovhd   : ~{(w_med*1e6 - cycles_to_us(c_med)):6.1f} us  (wall - hw)")
    print(f"    Throughput  : {throughput:6.1f} ops/s  (pulse-then-poll, single-thread)")

    return {
        "label": label,
        "cycles": cycles_list,
        "wall_s": wall_list,
        "cycle_median": c_med,
        "wall_median_s": w_med,
        "throughput_ops_s": throughput,
    }

In [ ]:
# Per-op bench wrappers — each returns the bench_op result dict

def bench_keygen(kem, n):
    print("--- KeyGen (random d, z per iter) ---")
    return bench_op(
        "KeyGen",
        lambda: kem.keygen(secrets.token_bytes(32), secrets.token_bytes(32)),
        n,
    )


def bench_encaps(kem, n):
    print("--- Encaps (warm pk, random m per iter) ---")
    pk_warm, _, _ = kem.keygen(secrets.token_bytes(32), secrets.token_bytes(32))
    return bench_op(
        "Encaps",
        lambda: kem.encaps(pk_warm, secrets.token_bytes(32)),
        n,
    )


def bench_decaps(kem, n):
    print("--- Decaps (warm sk, warm ct - match branch) ---")
    pk_warm, sk_warm, _ = kem.keygen(secrets.token_bytes(32), secrets.token_bytes(32))
    ct_warm, _, _ = kem.encaps(pk_warm, secrets.token_bytes(32))
    return bench_op(
        "Decaps",
        lambda: kem.decaps(sk_warm, ct_warm),
        n,
    )


def bench_full(kem, n):
    print("--- Full KEM round-trip (KG + Encaps + Decaps per iter) ---")

    def full_kem_once():
        d = secrets.token_bytes(32)
        z = secrets.token_bytes(32)
        m = secrets.token_bytes(32)
        pk, sk, c1 = kem.keygen(d, z)
        ct, ss1, c2 = kem.encaps(pk, m)
        ss2, c3 = kem.decaps(sk, ct)
        if ss1 != ss2:
            raise RuntimeError("Round-trip ss mismatch at runtime")
        return (c1 + c2 + c3,)

    return bench_op("Full KEM", full_kem_once, n)


OP_DISPATCH = {
    "keygen": bench_keygen,
    "encaps": bench_encaps,
    "decaps": bench_decaps,
    "full":   bench_full,
}

## Config — edit these and re-run cells below

In [ ]:
BITSTREAM_DIR = "/root/jupyter_notebooks/verilog_ML_KEM/bitstream"

bitfile = os.environ.get("ML_KEM_BIT", os.path.join(BITSTREAM_DIR, "ml_kem_bd.bit"))
n  = 100              # iterations per op (50-500 typical)
op = "all"            # one of: "all", "keygen", "encaps", "decaps", "full"

assert op in ("all", "keygen", "encaps", "decaps", "full"), f"unknown op: {op!r}"
assert os.path.exists(bitfile), f"bitfile not found on board: {bitfile}"
assert os.path.exists(bitfile.replace('.bit', '.hwh')), "missing .hwh next to .bit"
print(f"bitfile : {bitfile}")
print(f"n iters : {n}")
print(f"op      : {op}")


In [ ]:
# Open driver (run once per session). If you re-run this cell after a
# previous session, call kem.close() in the cleanup cell below first.
kem = MLKem768(bitfile)
print("Driver loaded.")

In [ ]:
# Run selected op(s). Results are kept in the `results` dict so you can
# inspect raw cycles/wall_s lists afterward (e.g. histogram plotting).
results = {}
if op == "all":
    for name in ["keygen", "encaps", "decaps", "full"]:
        results[name] = OP_DISPATCH[name](kem, n)
        print()
else:
    results[op] = OP_DISPATCH[op](kem, n)

## Optional: inspect raw distribution

After running the bench, `results[op_name]["cycles"]` and `results[op_name]["wall_s"]` hold the per-iteration samples. Useful for histogram plotting, percentile checks, or constant-time verification (decaps Δ=0 across many iters).

In [ ]:
# Example: print per-op summary in CSV-friendly form
for name, r in results.items():
    print(f"{name},"
          f"{int(r['cycle_median'])},"
          f"{cycles_to_us(r['cycle_median']):.2f},"
          f"{r['wall_median_s']*1e6:.2f},"
          f"{r['throughput_ops_s']:.2f}")

In [ ]:
# Cleanup: free CMA buffers. Skip if you intend to keep using `kem`.
kem.close()
print("Driver closed.")

## How to interpret results

- **`HW latency`** comes from the on-chip cycle counter (`REG_CYCLES`) and is the deterministic accelerator compute time. This number should match `tb_ml_kem_top` simulation cycle counts almost exactly.
- **`Wall time`** includes Python/PYNQ overhead: register writes for seeds + control + polling loop + CMA cache flush/invalidate. Typical overhead is ~50-100 µs per op.
- **`PYNQ overhead = wall - hw`** measures the software control-path cost. If this dominates, optimize the driver (e.g. coalesce register writes, use IRQ instead of poll); the RTL itself is already fine.
- **`Throughput`** is ops/s under single-threaded pulse-then-poll. To boost throughput further you would need either DMA-chained IRQ-driven flow or batched register writes.

## Reference cycle budget @ 100 MHz (from simulation)

| Op | HW cycles (sim) | HW latency |
|---|---:|---:|
| KeyGen | ~40k | ~400 µs |
| Encaps | ~57k | ~570 µs |
| Decaps | ~84k | ~840 µs |
| Full KEM | ~181k | ~1.8 ms |

Real on-board numbers from this notebook should match the HW columns within ~1% (deterministic) and add ~50-100 µs of PYNQ overhead per op.

---

## A vs C software-stack comparison

Two driver flavors:

- **Method A** (`MLKem768`): default Python `while True: read(STATUS)` polling.
- **Method C** (`MLKem768Fast`): identical except `_wait_done` is a compiled C function (`libmlkemfast.so`) called via `ctypes`. Saves ~1-5 µs per poll iteration → tens of µs to several ms per op.

Methods B (IRQ), D (separate AXI DMA), E (batched API) discussed in `docs/knowledge/perf-optimization-log.md` and skipped — see that doc for rationale.

### Build C lib on board (one-time)

Run from a board terminal:

```bash
cd /root/jupyter_notebooks/verilog_ML_KEM/Overlay
make
ls libmlkemfast.so   # should exist
```

If `make` is missing: `sudo apt-get install -y build-essential`.

In [ ]:
from ml_kem_driver import MLKem768Fast

# Re-run all four ops with the C-bypass driver. We re-open the overlay
# under MLKem768Fast (subclass of MLKem768) so the only flow difference
# is the _wait_done implementation.
n_compare = n  # match the A bench above for an apples-to-apples median

results_C = {}
with MLKem768Fast(bitfile) as kem_c:
    print(f"--- Method C (C-bypass busy-poll) — n={n_compare} ---\n")
    for name in ["keygen", "encaps", "decaps", "full"]:
        results_C[name] = OP_DISPATCH[name](kem_c, n_compare)
        print()

In [ ]:
# Side-by-side A vs C summary. `results` is from the A bench above.
print(f"{'Op':<8} {'Cyc(A)':>8} {'Cyc(C)':>8}  {'Wall A µs':>10} {'Wall C µs':>10}  "
      f"{'Δ wall':>9} {'Δ%':>6}  {'Tput A':>8} {'Tput C':>8}")
print("-" * 92)

for name in ["keygen", "encaps", "decaps", "full"]:
    if name not in results or name not in results_C:
        continue
    a = results[name]
    c = results_C[name]
    wa = a["wall_median_s"] * 1e6
    wc = c["wall_median_s"] * 1e6
    dwall = wc - wa
    dpct = (wc - wa) / wa * 100.0 if wa > 0 else 0.0
    print(f"{name:<8} "
          f"{int(a['cycle_median']):>8d} {int(c['cycle_median']):>8d}  "
          f"{wa:>10.1f} {wc:>10.1f}  "
          f"{dwall:>+9.1f} {dpct:>+5.1f}%  "
          f"{a['throughput_ops_s']:>8.1f} {c['throughput_ops_s']:>8.1f}")

print("\nCSV (op,cyc_A,cyc_C,wall_A_us,wall_C_us,delta_wall_us,delta_pct,tput_A,tput_C):")
for name in ["keygen", "encaps", "decaps", "full"]:
    if name not in results or name not in results_C:
        continue
    a = results[name]
    c = results_C[name]
    wa = a["wall_median_s"] * 1e6
    wc = c["wall_median_s"] * 1e6
    print(f"{name},{int(a['cycle_median'])},{int(c['cycle_median'])},"
          f"{wa:.2f},{wc:.2f},{wc-wa:+.2f},"
          f"{(wc-wa)/wa*100.0:+.2f},"
          f"{a['throughput_ops_s']:.2f},{c['throughput_ops_s']:.2f}")

---

## Method C-full — entire op flow in C

`MLKem768FullC` extends `MLKem768Fast` further: the per-op flow (write 17 seed/CTRL registers + start + busy-poll + cycles read) is one `libmlkemfast.so` call. **Cache flush()/invalidate() remain in Python** (Level 1).

Theoretical ceiling: **−25 to −45% wall vs Method A**. Eliminates the Python overhead of 17 mmio writes (~85-170 µs) on top of what Method C already removed (~33-41 µs polling). What remains in Python: cache mgmt (~100-200 µs), buffer copy, and final `bytes(...)` conversion.

Level 2 (cache via aarch64 DC instructions in C) is a future step if needed.

In [ ]:
from ml_kem_driver import MLKem768FullC

results_Cfull = {}
with MLKem768FullC(bitfile) as kem_cfull:
    print(f"--- Method C-full (entire flow in C) — n={n_compare} ---\n")
    for name in ["keygen", "encaps", "decaps", "full"]:
        results_Cfull[name] = OP_DISPATCH[name](kem_cfull, n_compare)
        print()

In [ ]:
# 3-way A vs C-poll vs C-full summary.
print(f"{'Op':<8} {'Wall A µs':>10} {'Wall C µs':>10} {'Wall Cf µs':>11}  "
      f"{'ΔC %':>7} {'ΔCf %':>8}  {'Tput A':>8} {'Tput C':>8} {'Tput Cf':>9}")
print("-" * 105)

for name in ["keygen", "encaps", "decaps", "full"]:
    if name not in results or name not in results_C or name not in results_Cfull:
        continue
    a  = results[name]
    c  = results_C[name]
    cf = results_Cfull[name]
    wa  = a ["wall_median_s"] * 1e6
    wc  = c ["wall_median_s"] * 1e6
    wcf = cf["wall_median_s"] * 1e6
    dC  = (wc  - wa) / wa * 100.0 if wa > 0 else 0.0
    dCf = (wcf - wa) / wa * 100.0 if wa > 0 else 0.0
    print(f"{name:<8} "
          f"{wa:>10.1f} {wc:>10.1f} {wcf:>11.1f}  "
          f"{dC:>+6.1f}% {dCf:>+7.1f}%  "
          f"{a['throughput_ops_s']:>8.1f} {c['throughput_ops_s']:>8.1f} "
          f"{cf['throughput_ops_s']:>9.1f}")

print("\nCSV (op,wall_A_us,wall_C_us,wall_Cfull_us,deltaC_pct,deltaCf_pct,tput_A,tput_C,tput_Cf):")
for name in ["keygen", "encaps", "decaps", "full"]:
    if name not in results or name not in results_C or name not in results_Cfull:
        continue
    a  = results[name]
    c  = results_C[name]
    cf = results_Cfull[name]
    wa  = a ["wall_median_s"] * 1e6
    wc  = c ["wall_median_s"] * 1e6
    wcf = cf["wall_median_s"] * 1e6
    print(f"{name},{wa:.2f},{wc:.2f},{wcf:.2f},"
          f"{(wc-wa)/wa*100.0:+.2f},{(wcf-wa)/wa*100.0:+.2f},"
          f"{a['throughput_ops_s']:.2f},{c['throughput_ops_s']:.2f},"
          f"{cf['throughput_ops_s']:.2f}")

---

## Method B — IRQ-driven wait

`MLKem768IRQ` replaces the polling loop with a kernel UIO interrupt wake-up
via PYNQ's `Interrupt` class. RTL drives `irq_done = status_done` (level-high
while op complete). Linux UIO handles edge detect + masking; PYNQ wraps in
asyncio. The driver hides asyncio behind `asyncio.run` so the API matches A.

Requires the bitstream variant where `ml_kem_top_0/irq_done` is wired to
`zynq_ultra_ps_e_0/pl_ps_irq0[0]` via xlconcat. Built in R-new-D K2.

**Expected**: comparable wall-time to A/C for sub-ms ops (Linux IRQ-to-user
latency ~50-200 µs ≈ polling overhead). Real win is CPU availability during
HW execution — not directly visible in single-thread bench.

In [ ]:
from ml_kem_driver import MLKem768IRQ

results_B = {}
with MLKem768IRQ(bitfile) as kem_b:
    print(f"--- Method B (IRQ-driven via PYNQ Interrupt) — n={n_compare} ---\n")
    for name in ["keygen", "encaps", "decaps", "full"]:
        results_B[name] = OP_DISPATCH[name](kem_b, n_compare)
        print()

In [ ]:
# 4-way summary: A (poll) vs B (IRQ) vs C (C-poll) vs C-full (all-C).
print(f"{'Op':<8} "
      f"{'Wall A µs':>10} {'Wall B µs':>10} {'Wall C µs':>10} {'Wall Cf µs':>11}  "
      f"{'ΔB %':>6} {'ΔC %':>6} {'ΔCf %':>7}  "
      f"{'Tput A':>8} {'Tput B':>8} {'Tput C':>8} {'Tput Cf':>9}")
print("-" * 130)

for name in ["keygen", "encaps", "decaps", "full"]:
    if not all(name in r for r in (results, results_B, results_C, results_Cfull)):
        continue
    a  = results[name]
    b  = results_B[name]
    c  = results_C[name]
    cf = results_Cfull[name]
    wa, wb, wc, wcf = (r["wall_median_s"] * 1e6 for r in (a, b, c, cf))
    dB  = (wb  - wa) / wa * 100.0 if wa > 0 else 0.0
    dC  = (wc  - wa) / wa * 100.0 if wa > 0 else 0.0
    dCf = (wcf - wa) / wa * 100.0 if wa > 0 else 0.0
    print(f"{name:<8} "
          f"{wa:>10.1f} {wb:>10.1f} {wc:>10.1f} {wcf:>11.1f}  "
          f"{dB:>+5.1f}% {dC:>+5.1f}% {dCf:>+6.1f}%  "
          f"{a['throughput_ops_s']:>8.1f} {b['throughput_ops_s']:>8.1f} "
          f"{c['throughput_ops_s']:>8.1f} {cf['throughput_ops_s']:>9.1f}")

print("\nCSV (op,wall_A_us,wall_B_us,wall_C_us,wall_Cfull_us,deltaB_pct,deltaC_pct,deltaCf_pct,tput_A,tput_B,tput_C,tput_Cf):")
for name in ["keygen", "encaps", "decaps", "full"]:
    if not all(name in r for r in (results, results_B, results_C, results_Cfull)):
        continue
    a  = results[name]
    b  = results_B[name]
    c  = results_C[name]
    cf = results_Cfull[name]
    wa, wb, wc, wcf = (r["wall_median_s"] * 1e6 for r in (a, b, c, cf))
    print(f"{name},{wa:.2f},{wb:.2f},{wc:.2f},{wcf:.2f},"
          f"{(wb-wa)/wa*100.0:+.2f},{(wc-wa)/wa*100.0:+.2f},{(wcf-wa)/wa*100.0:+.2f},"
          f"{a['throughput_ops_s']:.2f},{b['throughput_ops_s']:.2f},"
          f"{c['throughput_ops_s']:.2f},{cf['throughput_ops_s']:.2f}")